# CS4.406: Information Retrieval & Extraction — Assignment 2
## Learning from Click-Logs on EB-NeRD and MIND

**Team Assignment (Groups of 2)**  
**Course:** CS4.406 Information Retrieval & Extraction  
**Competitions:**
- [MIND Codabench Competition](https://www.codabench.org/competitions/13967/)
- [RecSys 2024 Challenge Codabench](https://www.codabench.org/competitions/2469/)

---

### Pipeline Overview:
1. **Stage 1 (Retrieval):** Popularity-based candidate generation
2. **Feature Engineering (Q1):** User click history, recency decay ($w = 2^{-\Delta t / 24\text{h}}$), category matching, session dynamics, and position bias with strict temporal boundary enforcement
3. **Stage 2 (Re-Ranker - Q2):** LightGBM GBDT binary ranker trained on click-logs
4. **Ablation Study (Q3):** Category-aware user interest matching with paired bootstrap 95% CI
5. **Serving & Scale Analysis (Q4):** Memory footprint, p99 retrieval latency, cost/QPS, 10x scaling breakdown
6. **Extended Evaluation (Q5):** AUC, MRR, nDCG@5, nDCG@10, Novelty, Diversity, Coverage with Cold/Warm and Head/Tail slicing
7. **Anti-Gaming (Q9):** Temporal boundary verification and future-click leakage tests
8. **Submission Generation:** Memory-efficient prediction batching and Codabench zip creation

In [15]:
ls -al

total 28
drwxr-xr-x 4 root root 4096 Sep 20 11:12 ./
drwxr-xr-x 4 root root 4096 Sep 20 11:09 ../
-rw-r--r-- 1 root root  101 Sep 20  2026 requirements.txt
-rw-r--r-- 1 root root 6810 Sep 20  2026 run_pipeline.py
drwxr-xr-x 3 root root 4096 Sep 20  2026 src/
drwxr-xr-x 3 root root 4096 Sep 20  2026 tests/


## 1. Environment Setup & Dependencies

In [16]:
# Install required packages
!pip install -q polars pyarrow lightgbm scikit-learn scipy tqdm fpdf2 huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 7.4 MB/s eta 0:00:00


## 2. Mount Google Drive or Clone Repository
If you have uploaded the `a2` directory to your Google Drive, mount it and cd into it.  
Alternatively, clone your git repository.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust path to where you placed the a2 folder in your Google Drive:
%cd /content/drive/MyDrive/IRE/a2
!ls -la

Mounted at /content/drive
/content/drive/MyDrive/IRE/a2
total 28
drwx------ 5 root root 4096 Sep 20 09:45 .
drwx------ 3 root root 4096 Sep 20 09:45 ..
drwx------ 2 root root 4096 Sep 20 09:52 data
-rw------- 1 root root  101 Sep 20 08:41 requirements.txt
-rw------- 1 root root 6810 Sep 20 09:29 run_pipeline.py
drwx------ 2 root root 4096 Sep 20 09:46 src
drwx------ 2 root root 4096 Sep 20 09:46 tests


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Dataset Download

### EB-NeRD Download (Public S3)
EB-NeRD demo (21.5MB) for development or large for final Codabench test submission.

In [14]:
rm -rf outputs/

In [ ]:
rm -rf data/

In [ ]:
rm -rf models/

In [17]:
# Download EB-NeRD Demo (for fast dev/testing):
!python -m src.data_loader --dataset ebnerd --size demo

# When ready for Codabench test set:
!python -m src.data_loader --dataset ebnerd --size large

 IRE Assignment 2 — Data Pipeline
ebnerd_demo: 21.5MB [00:01, 12.1MB/s]                
  Extracting ebnerd_demo...
  Extracted to /content/drive/MyDrive/IRE/a2/data/raw/ebnerd/ebnerd_demo

🔧 Building EB-NeRD feature store...
  Articles: 11,777
  Train behaviors: 24,724 impressions
  Validation behaviors: 25,356 impressions
  User histories: 1,590 users
  Popularity: 1,114 articles with clicks
✅ EB-NeRD feature store built

 Data pipeline complete!
 IRE Assignment 2 — Data Pipeline
ebnerd_large: 3.19GB [01:36, 33.1MB/s]                
  Extracting ebnerd_large...
  Extracted to /content/drive/MyDrive/IRE/a2/data/raw/ebnerd/ebnerd_large
ebnerd_testset: 1.63GB [00:49, 33.1MB/s]                
  Extracting ebnerd_testset...
  Extracted to /content/drive/MyDrive/IRE/a2/data/raw/ebnerd/ebnerd_testset

🔧 Building EB-NeRD feature store...
  Articles: 125,541
  Train behaviors: 12,063,890 impressions
  Validation behaviors: 12,566,385 impressions
  User histories: 788,090 users
  Popularity:

### MIND Download (Hugging Face)
For MIND, authenticate with your free Hugging Face token (the dataset is gated):

In [18]:
# Log in with your free Hugging Face access token (from https://huggingface.co/settings/tokens):
from huggingface_hub import login
login()

# Download MIND dataset using the modern 'hf' CLI:
!hf download yjw1029/MIND --repo-type dataset --local-dir data/raw/mind

# Build MIND feature store (automatically detects downloaded zips or downloads with HF token):
!python -m src.data_loader --dataset mind --size small

Hint: A new version of huggingface_hub (1.32.0) is available! You are using version 1.29.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0% 0/7 [00:00<?, ?it/s]Still waiting to acquire lock on /content/drive/MyDrive/IRE/a2/data/raw/mind/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

Reconstructing (incomplete total...):   0% 0.00/1.94k [00:00<?, ?B/s]         

Fetching 7 files:  14% 1/7 [00:00<00:01,  4.70it/s]
Reconstructing (incomplete total...):   0% 1.94k/605M [00:00<18:19:44, 9.16kB/s]
Reconstructing (incomplete total...):   0% 1.94k/792M [00:00<24:00:31, 9.16kB/s]
Reconstructing (incomplete total...):   0% 1.94k/1.32G [00:00<40:04:54, 9.16kB/s]
Reconstructing (incomplete total...):   0% 1.94k/1.32G [00:00<40:04:54, 9.16kB/s]
Reconstructing (incomplete total...):   0

In [19]:
!python -m src.data_loader --dataset mind --size test

 IRE Assignment 2 — Data Pipeline
  Extracting MINDlarge_test...
  Unwrapping nested directory: MINDlarge_test -> MINDlarge_test
  Extracted to /content/drive/MyDrive/IRE/a2/data/raw/mind/MINDlarge_test
 Data pipeline complete!


## 4. Run End-to-End Pipeline (Q1 - Q5)
This runs the complete workflow in one command:
- Feature engineering with behavioural boundary enforcement
- Candidate generation
- LightGBM re-ranker training
- Baseline reproduction vs. Improved model vs. Ablation
- Paired bootstrap 95% confidence intervals
- Serving & scale analysis (latency, memory, QPS, 10x breakdown)
- Anti-gaming tests

In [26]:
# Run pipeline on EB-NeRD (demo):
!python run_pipeline.py --dataset ebnerd --size demo --skip-download

 IRE Assignment 2 — Full Pipeline
 Dataset: EBNERD, Size: demo

⏭️  Skipping download (--skip-download)

════════════════════════════════════════════════════════════
 Processing: EBNERD
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
 STEPS 2-3: Ablation Study (EBNERD)
────────────────────────────────────────────────────────────

📐 Building training features...
Building features: 100% 25000/25000 [00:18<00:00, 1337.51it/s]

📐 Building validation features...
Building features: 100% 25000/25000 [00:24<00:00, 1016.84it/s]

 Ablation Study: EBNERD
Candidate generation: 100% 25000/25000 [00:00<00:00, 129348.27it/s]

 Evaluation: POPULARITY_BASELINE on EBNERD

  📊 Overall
    AUC         : 0.4235  [0.4213, 0.4256]  (n=25,000)
    MRR         : 0.2653  [0.2623, 0.2685]  (n=25,000)
    nDCG@5      : 0.2882  [0.2845, 0.2921]  (n=25,000)
    nDCG@10     : 0.3877  [0.3847, 0.3910]  (n=25,000)
    Novelty     : 4.0650  [4.0

In [21]:
# Run pipeline on MIND (small):
!python run_pipeline.py --dataset mind --size small --skip-download

 IRE Assignment 2 — Full Pipeline
 Dataset: MIND, Size: small

⏭️  Skipping download (--skip-download)

════════════════════════════════════════════════════════════
 Processing: MIND
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
 STEPS 2-3: Ablation Study (MIND)
────────────────────────────────────────────────────────────

📐 Building training features...
Building features: 100% 156965/156965 [00:35<00:00, 4451.86it/s]

📐 Building validation features...
Building features: 100% 73152/73152 [00:15<00:00, 4857.81it/s]

 Ablation Study: MIND
Candidate generation: 100% 73152/73152 [00:01<00:00, 58711.77it/s]

 Evaluation: POPULARITY_BASELINE on MIND

  📊 Overall
    AUC         : 0.5318  [0.5301, 0.5335]  (n=73,152)
    MRR         : 0.2666  [0.2645, 0.2690]  (n=73,152)
    nDCG@5      : 0.2454  [0.2430, 0.2479]  (n=73,152)
    nDCG@10     : 0.3092  [0.3070, 0.3115]  (n=73,152)
    Novelty     : 8.8415  [8.8122, 8.8

## 5. Anti-Gaming Verification (Q9)
Assert temporal boundary enforcement: training impressions strictly precede validation impressions.

In [22]:
!python tests/test_no_leakage.py

 Anti-Gaming Tests (Q9)
  ✅ MIND: No leakage. Train max=2019-11-09 09:59:58, Dev min=2019-11-15 10:00:00
  ✅ EB-NeRD: No leakage. Train max=2023-05-25 06:59:59, Val min=2023-05-25 07:00:00
  ✅ Feature engineering uses only past clicks (by construction):
     - User histories from training split only
     - Session counters accumulated in temporal order
     - Popularity index from training data only
     - No future information accessed in compute_features_for_impression()
  ✅ Test set has no click labels (by construction)
     Predictions use only: popularity, category, freshness, user history
     No features unavailable at serving time are used

 All anti-gaming tests passed!


## 6. Generate Codabench Predictions & Zip Files
Generates predictions using batch processing (memory-safe for large test sets) and packages them into upload-ready `.zip` files containing `prediction.txt`.

In [28]:
# Generate EB-NeRD predictions with trained LightGBM re-ranker
!python -m src.predict --dataset ebnerd --method reranker

  📂 Loaded model from /content/drive/MyDrive/IRE/a2/models/reranker_ebnerd_full.txt

🔮 Generating EB-NeRD predictions (reranker)...
  ⚠️  Test behaviors not found under /content/drive/MyDrive/IRE/a2/data/raw/ebnerd/ebnerd_testset

⚠️  Skipped zip creation: /content/drive/MyDrive/IRE/a2/outputs/ebnerd_prediction_reranker.txt was not generated.


In [24]:
# Generate MIND predictions with trained LightGBM re-ranker
!python -m src.predict --dataset mind --method reranker

  📂 Loaded model from /content/drive/MyDrive/IRE/a2/models/reranker_mind_full.txt

🔮 Generating MIND predictions (reranker)...
    50,000 written...
    100,000 written...
    150,000 written...
    200,000 written...
    250,000 written...
    300,000 written...
    350,000 written...
    400,000 written...
    450,000 written...
    500,000 written...
    550,000 written...
    600,000 written...
    650,000 written...
    700,000 written...
    750,000 written...
    800,000 written...
    850,000 written...
    900,000 written...
    950,000 written...
    1,000,000 written...
    1,050,000 written...
    1,100,000 written...
    1,150,000 written...
    1,200,000 written...
    1,250,000 written...
    1,300,000 written...
    1,350,000 written...
    1,400,000 written...
    1,450,000 written...
    1,500,000 written...
    1,550,000 written...
    1,600,000 written...
    1,650,000 written...
    1,700,000 written...
    1,750,000 written...
    1,800,000 written...
    1,850,00

### Download Prediction Zips to Local Machine for Codabench Upload

In [ ]:
from google.colab import files

# Download predictions:
import os
for f in os.listdir('outputs'):
    if f.endswith('.zip'):
        print(f'Downloading {f}...')
        files.download(os.path.join('outputs', f))

print('Upload these zips to the respective Codabench leaderboards:')
print('MIND: https://www.codabench.org/competitions/13967/')
print('RecSys 2024: https://www.codabench.org/competitions/2469/')

## 7. Inspect Evaluation Summaries & Results

In [25]:
import json
from pathlib import Path

summary_file = Path('outputs/all_eval_summary.json')
if summary_file.exists():
    with open(summary_file) as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2))
else:
    print('No summary file found yet. Run the pipeline first.')

{
  "mind_popularity_baseline": {
    "dataset": "mind",
    "method": "popularity_baseline",
    "overall": {
      "AUC": {
        "mean": 0.531751528369352,
        "CI_lower": 0.5300714381653492,
        "CI_upper": 0.5335031104464891,
        "n": 73152
      },
      "MRR": {
        "mean": 0.26661218014862703,
        "CI_lower": 0.26445641410610904,
        "CI_upper": 0.2689916719012552,
        "n": 73152
      },
      "nDCG@5": {
        "mean": 0.2453715239294527,
        "CI_lower": 0.2429812448939834,
        "CI_upper": 0.24788061409165246,
        "n": 73152
      },
      "nDCG@10": {
        "mean": 0.30916265663215475,
        "CI_lower": 0.3069959617366559,
        "CI_upper": 0.31152238043955244,
        "n": 73152
      },
      "Novelty": {
        "mean": 8.841498205939303,
        "CI_lower": 8.812230100477558,
        "CI_upper": 8.873472498595614,
        "n": 73152
      },
      "Coverage": {
        "value": 0.03091756338330421
      }
    },
    "cold_